In [ ]:
!pip install tf_keras==2.18.0

In [ ]:
import numpy as np
import cv2
import PIL.Image as Image
import os
import matplotlib.pylab as plt
import tensorflow as tf
import tensorflow_hub as hub
import tarfile
import pathlib

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.metrics import confusion_matrix


download data from the flower dataset , online and extract

In [ ]:



dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
download_path = tf.keras.utils.get_file(fname="flower_photos.tgz", origin=dataset_url, cache_dir="/content", extract=False)


extract_path = "/content/flower_photos"

try:
    with tarfile.open(download_path, "r:gz") as tar:
        tar.extractall(path="/content")  # Extract to /content, so the folder appears as /content/flower_photos
    print(" Dataset extracted successfully.")
except Exception as e:
    print(" Extraction failed:", e)


data_dir = pathlib.Path(extract_path)


if not data_dir.exists():
    print(f" Extraction path does not exist: {extract_path}")
else:
    print(f" Extraction path exists: {extract_path}")
    print(" Folders inside:", os.listdir(extract_path))


image_list = list(data_dir.glob('*/*.jpg'))

if image_list:
    print(f" Found {len(image_list)} images.")
    print(" Sample images:", image_list[:5])
else:
    print(f" No images found! Check extraction path: {extract_path}")


/tmp/ipykernel_14665/1122345463.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="/content")  # Extract to /content, so the folder appears as /content/flower_photos


 Dataset extracted successfully.
 Extraction path exists: /content/flower_photos
 Folders inside: ['tulips', 'sunflowers', 'LICENSE.txt', 'daisy', 'dandelion', 'roses']
 Found 3670 images.
 Sample images: [PosixPath('/content/flower_photos/tulips/17324469461_2b318aff8d_m.jpg'), PosixPath('/content/flower_photos/tulips/18245124970_e68fd3f3c3.jpg'), PosixPath('/content/flower_photos/tulips/14053292975_fdc1093571_n.jpg'), PosixPath('/content/flower_photos/tulips/16074109313_2cc14c7d16.jpg'), PosixPath('/content/flower_photos/tulips/7205698252_b972087cc2.jpg')]


another method of download and extract flower data from

In [ ]:
list(data_dir.glob('*/*.jpg'))[:5]

[PosixPath('/content/flower_photos/tulips/17324469461_2b318aff8d_m.jpg'),
 PosixPath('/content/flower_photos/tulips/18245124970_e68fd3f3c3.jpg'),
 PosixPath('/content/flower_photos/tulips/14053292975_fdc1093571_n.jpg'),
 PosixPath('/content/flower_photos/tulips/16074109313_2cc14c7d16.jpg'),
 PosixPath('/content/flower_photos/tulips/7205698252_b972087cc2.jpg')]

In [ ]:
image_count = len(list(data_dir.glob('*/*.jpg')))
print(image_count)

3670


In [ ]:
roses = list(data_dir.glob('roses/*'))
roses[:5]

[PosixPath('/content/flower_photos/roses/15951588433_c0713cbfc6_m.jpg'),
 PosixPath('/content/flower_photos/roses/509239741_28e2cfe492_m.jpg'),
 PosixPath('/content/flower_photos/roses/5799616059_0ffda02e54.jpg'),
 PosixPath('/content/flower_photos/roses/6363951285_a802238d4e.jpg'),
 PosixPath('/content/flower_photos/roses/2093263381_afd51358a3.jpg')]

In [ ]:
flowers_images_dict = {
    'roses': list(data_dir.glob('roses/*')),
    'daisy': list(data_dir.glob('daisy/*')),
    'dandelion': list(data_dir.glob('dandelion/*')),
    'sunflowers': list(data_dir.glob('sunflowers/*')),
    'tulips': list(data_dir.glob('tulips/*')),
}

In [ ]:
flowers_labels_dict = {
    'roses': 0,
    'daisy': 1,
    'dandelion': 2,
    'sunflowers': 3,
    'tulips': 4,
}

In [ ]:
X, y = [], []

for flower_name, images in flowers_images_dict.items():
    for image in images:
        img = cv2.imread(str(image))
        resized_img = cv2.resize(img,(224,224))
        X.append(resized_img)
        y.append(flowers_labels_dict[flower_name])

In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

In [ ]:
X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

retrain model using the flower data


In [ ]:
import tf_keras as keras
import tensorflow_hub as hub

feature_extractor_model = "https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2"

pretrained_model_without_top_layer = hub.KerasLayer(
    feature_extractor_model, input_shape=(224, 224, 3), trainable=False)



In [ ]:

num_of_flowers = 5

model = keras.Sequential([
    pretrained_model_without_top_layer,
    keras.layers.Dense(num_of_flowers)
])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 1280)              3413024   
                                                                 
 dense (Dense)               (None, 5)                 6405      
                                                                 
Total params: 3419429 (13.04 MB)
Trainable params: 6405 (25.02 KB)
Non-trainable params: 3413024 (13.02 MB)
_________________________________________________________________


In [ ]:
model.compile(
  optimizer="adam",
  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=['acc'])

model.fit(X_train_scaled, y_train, epochs=5)


In [ ]:
loss, acc = model.evaluate(X_test_scaled, y_test)
print("Test Accuracy:", acc)

In [ ]:
predictions = model.predict(X_test_scaled[:5])
predicted_classes = np.argmax(predictions, axis=1)
print(predicted_classes)